In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
n_planktons = 50
arena_length = 100
n_frames = 100
Dr = 3
Dt = 10
delta_t = 0.1
velocity = 3

In [ ]:
Kb = 1.38e-23
T = 300
eta = 1e-3
R = 1000e-6

real_Dr = Kb*T/(8 * np.pi * eta * R**3)
real_Dt = Kb*T/(6 * np.pi * eta * R)

print("Dr", real_Dr)
print("Dt", real_Dt)

In [ ]:
Dr = real_Dr
Dt = real_Dt

In [ ]:
velocity = 10
target_angle = np.pi / 2 # 90 degrees 
response_angle = 180 * (np.pi / 180) # 5 degrees

In [ ]:
x = np.random.rand(n_planktons) * 2 * arena_length - arena_length
y = np.random.rand(n_planktons) * 2 * arena_length - arena_length
phi = np.random.rand(n_planktons) * 2 * np.pi

# History of positions (used for trails)
trail_length = 20
history_x = np.full((trail_length, n_planktons), np.nan)
history_y = np.full((trail_length, n_planktons), np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
scat = ax.scatter(x, y, c='white', s=20, alpha=0.5)
trails = [ax.plot([], [], '-', linewidth=1, alpha=0.5)[0] for _ in range(n_planktons)]
ax.set_xlim(-arena_length, arena_length)
ax.set_ylim(-arena_length, arena_length)
ax.set_facecolor('black')

In normal conditions the orientation, phi, is controlled by the rotational diffusion co-efficient.

But, when the light is turned on, phi is also adjusted independently.

In [ ]:
def update(frame):
    global x, y, phi, history_x, history_y
    
    phi = target_angle + response_angle * (2 * np.random.rand(n_planktons) - 1)
    # phi = phi + np.sqrt(2 * Dr * delta_t) * np.random.randn(n_planktons)
        
    x = x + velocity * np.cos(phi) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)
    y = y + velocity * np.sin(phi) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)

    # Reflect at walls
    y[y > arena_length] = 2 * arena_length - y[y > arena_length]
    y[y < -arena_length] = -2 * arena_length - y[y < -arena_length]
    x[x > arena_length] = 2 * arena_length - x[x > arena_length]
    x[x < -arena_length] = -2 * arena_length - x[x < -arena_length]

    scat.set_offsets(np.c_[x, y])

    # Update trails
    history_x = np.roll(history_x, -1, axis=0)
    history_y = np.roll(history_y, -1, axis=0)
    history_x[-1, :] = x
    history_y[-1, :] = y

    for i, trail in enumerate(trails):
        trail.set_data(history_x[:, i], history_y[:, i])

In [ ]:
ani = FuncAnimation(fig, update, frames=100, interval=100)
HTML(ani.to_jshtml())

Notes:

- Orientation contributes more than the veloctity for upwelling of plankton at the surface. As long as the orientation is upward, even with minium velocity they'll reach up at some point of time.